# GTFN — orijinal protokol, test sızıntısı olmadan

**Ne değişti?** Eğitim kümesi, model ve eğitim orijinal GTFN ile birebir aynı. Tek fark, raporlanan epoch'un **test kümesine göre değil, ayrı bir doğrulama kümesine göre** seçilmesi.
Doğrulama pikselleri eğitim dışındaki piksellerden alınıyor (sınıf başına en fazla 10) ve test kümesinden çıkarılıyor.

Her koşu için üç sonuç kaydediliyor:
- `selected`: doğrulamaya göre seçilen epoch'un test sonucu → **raporlanacak sonuç**
- `last_epoch`: son epoch'un test sonucu
- `test_oracle_leaky`: orijinal yöntem (teste göre seçim) → sadece sızıntının büyüklüğünü ölçmek için

Sonuçlar Drive'a (`hsi-gnn-PHD/results/GTFN_noleak/`) ve W&B'ye yazılır.

**Sıra:** Hücreleri yukarıdan aşağıya çalıştır. Kesinti olursa 1–5 arası hücreleri yeniden çalıştırıp 7. hücreyle devam et; tamamlanmış koşular atlanır.

Her çalıştırma, Drive'daki sonuç klasöründe `DENEY_KAYDI.md` dosyasına otomatik bir giriş ekler. Deneyin amacını ve yorumunu repodaki `DENEYLER.md` dosyasına yaz.

## 1. GPU kontrolü

In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-b745c899-c9d0-ddb1-de8a-247838097e28)


## 2. Drive'ı bağla

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Kodu GitHub'dan çek
`GITHUB_USER` satırına kendi kullanıcı adını yaz.

In [3]:
GITHUB_USER = "esratepe"   # <-- değiştir

import os
os.chdir("/content")
if os.path.exists("/content/GTFN/.git"):
    %cd /content/GTFN
    !git pull -q
else:
    !git clone -q https://github.com/{GITHUB_USER}/GTFN.git /content/GTFN
    %cd /content/GTFN
!git log --oneline -3
!ls scripts run_utils.py

/content/GTFN
e8275af (HEAD -> master, origin/master, origin/HEAD) Test sızıntısız epoch seçimi, Drive/W&B kaydı, çoklu seed çalıştırıcısı ve deney günlüğü
c727270 Veri yolları göreli yapıldı, data/ gitignore'a eklendi
36ac947 GTFN
run_utils.py

scripts:
run_seeds.py


## 4. Veriyi Drive'dan kopyala

In [4]:
import os, shutil
SRC = "/content/drive/MyDrive/hsi-gnn-PHD/HSI_data"
os.makedirs("data", exist_ok=True)
for f in ["Indian_pines_corrected.mat", "Indian_pines_gt.mat", "PaviaU.mat",
          "PaviaU_gt.mat", "Salinas_corrected.mat", "Salinas_gt.mat"]:
    shutil.copy(os.path.join(SRC, f), "data")
    print("✓", f, round(os.path.getsize(os.path.join("data", f)) / 2**20, 1), "MB")

✓ Indian_pines_corrected.mat 5.7 MB
✓ Indian_pines_gt.mat 0.0 MB
✓ PaviaU.mat 33.2 MB
✓ PaviaU_gt.mat 0.0 MB
✓ Salinas_corrected.mat 25.3 MB
✓ Salinas_gt.mat 0.0 MB


## 5. Paketler ve W&B girişi
Anahtarı Colab Secrets'a `WANDB_API_KEY` adıyla kaydettiysen oradan okunur. Okunamazsa `wandb.login()` anahtarı sorar: wandb.ai/authorize sayfasından kopyalayıp yapıştır.

In [5]:
!pip install -q einops wandb
import os, wandb
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("Anahtar Colab Secrets'tan alındı.")
except Exception as e:
    print("Secrets okunamadı, anahtar elle istenecek:", type(e).__name__)
wandb.login()

Secrets okunamadı, anahtar elle istenecek: TimeoutException


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: esratepe (esratepe-i-s) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 6. Kısa deneme (Indian, seed 0, 20 epoch) — yaklaşık 1 dakika
Ayrı bir klasöre yazar, asıl sonuçlara karışmaz.

In [6]:
!python -u scripts/run_seeds.py \
    --out_root /content/drive/MyDrive/hsi-gnn-PHD/results/GTFN_noleak_deneme \
    --datasets Indian --seeds 0 --epoches 20 --wandb_project hsi-gtfn-deneme \
    --exp_id E002-deneme --note "Kurulum testi: 20 epoch, tek seed"

commit=e8275af-ep20  seeds=[0]  out_root=/content/drive/MyDrive/hsi-gnn-PHD/results/GTFN_noleak_deneme

Indian  {"patches": 9, "n_gcn": 21, "pca_band": 70, "epoches": 20}
  seed= 0  çalışıyor... OA=87.69  (son epoch 87.69, sızıntılı 87.69)  34 sn

ÖZET (ortalama ± std, test kümesi)
Indian   n= 1  OA=87.69±0.00  AA=92.50±0.00  κ=86.00±0.00   | son epoch OA=87.69  sızıntılı OA=87.69

Özet kaydedildi: summary_e8275af-ep20.json
Deney kaydına eklendi: /content/drive/MyDrive/hsi-gnn-PHD/results/GTFN_noleak_deneme/DENEY_KAYDI.md


## 7. Tam deney: 3 veri seti × 10 seed × 200 epoch
Tahmini süre T4'te yaklaşık 3 saat (Indian ~30 dk, PaviaU ~1 sa, Salinas ~1,5 sa).
Bu sürede bilgisayar uyku moduna geçmemeli. Bağlantı koparsa 1–5'i yeniden çalıştırıp bu hücreyi tekrar başlat: bitmiş seed'ler atlanır.

In [7]:
!python -u scripts/run_seeds.py \
    --out_root /content/drive/MyDrive/hsi-gnn-PHD/results/GTFN_noleak \
    --datasets Indian PaviaU Salinas --seeds 0-9 --wandb_project hsi-gtfn \
    --exp_id E002 --note "Orijinal protokol, test sızıntısız seçim"

commit=e8275af  seeds=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  out_root=/content/drive/MyDrive/hsi-gnn-PHD/results/GTFN_noleak

Indian  {"patches": 9, "n_gcn": 21, "pca_band": 70, "epoches": 200}
  seed= 0  çalışıyor... OA=93.45  (son epoch 93.78, sızıntılı 93.78)  195 sn
  seed= 1  çalışıyor... OA=93.53  (son epoch 92.42, sızıntılı 93.71)  189 sn
  seed= 2  çalışıyor... OA=91.52  (son epoch 91.52, sızıntılı 92.57)  186 sn
  seed= 3  çalışıyor... OA=94.22  (son epoch 94.22, sızıntılı 94.85)  187 sn
  seed= 4  çalışıyor... OA=92.82  (son epoch 92.72, sızıntılı 92.82)  187 sn
  seed= 5  çalışıyor... OA=92.91  (son epoch 93.20, sızıntılı 93.61)  187 sn
  seed= 6  çalışıyor... OA=94.34  (son epoch 93.15, sızıntılı 94.34)  189 sn
  seed= 7  çalışıyor... OA=94.96  (son epoch 94.55, sızıntılı 94.96)  189 sn
  seed= 8  çalışıyor... OA=91.69  (son epoch 91.44, sızıntılı 91.44)  194 sn
  seed= 9  çalışıyor... OA=91.80  (son epoch 92.32, sızıntılı 92.61)  194 sn

PaviaU  {"patches": 9, "n_gcn": 15, "pca_b

## 8. Eski (sızıntılı) sonuçlarla karşılaştırma
Aynı seed'de eğitim kümesi birebir aynı olduğu için seed seed karşılaştırma yapılabilir.

In [4]:
import json, glob, os
import numpy as np
OLD = "/content/drive/MyDrive/hsi-gnn-PHD/GTFN/output"
NEW = "/content/drive/MyDrive/hsi-gnn-PHD/results/GTFN_noleak"

for ds in ["Indian", "PaviaU", "Salinas"]:
    rows = []
    for seed in range(10):
        old_f = os.path.join(OLD, f"{ds}_seed{seed:02d}.json")
        new_f = sorted(glob.glob(os.path.join(NEW, ds, f"seed{seed:02d}_*", "metrics.json")))
        if not (os.path.exists(old_f) and new_f):
            continue
        old = json.load(open(old_f))
        new = json.load(open(new_f[-1]))
        rows.append((seed, old["OA"], new["test_oracle_leaky"]["OA"], new["selected"]["OA"], new["last_epoch"]["OA"]))
    if not rows:
        continue
    a = np.array(rows)
    print(f"\n{ds}  (n={len(a)})")
    print(" seed | eski OA | yeni-sızıntılı | yeni-seçili (val) | son epoch")
    for r in rows:
        print(f"  {r[0]:3d} |  {r[1]:6.2f} |     {r[2]:6.2f}     |      {r[3]:6.2f}       |  {r[4]:6.2f}")
    print(f" ort. |  {a[:,1].mean():6.2f} |     {a[:,2].mean():6.2f}     |      {a[:,3].mean():6.2f}       |  {a[:,4].mean():6.2f}")
    print(f" sızıntı farkı (eski − val seçili): {np.mean(a[:,1]-a[:,3]):+.2f} puan")


Indian  (n=10)
 seed | eski OA | yeni-sızıntılı | yeni-seçili (val) | son epoch
    0 |   94.18 |      93.78     |       93.45       |   93.78
    1 |   93.75 |      93.71     |       93.53       |   92.42
    2 |   92.25 |      92.57     |       91.52       |   91.52
    3 |   94.64 |      94.85     |       94.22       |   94.22
    4 |   93.16 |      92.82     |       92.82       |   92.72
    5 |   93.49 |      93.61     |       92.91       |   93.20
    6 |   94.08 |      94.34     |       94.34       |   93.15
    7 |   94.44 |      94.96     |       94.96       |   94.55
    8 |   91.66 |      91.44     |       91.69       |   91.44
    9 |   92.38 |      92.61     |       91.80       |   92.32
 ort. |   93.40 |      93.47     |       93.12       |   92.93
 sızıntı farkı (eski − val seçili): +0.28 puan

PaviaU  (n=10)
 seed | eski OA | yeni-sızıntılı | yeni-seçili (val) | son epoch
    0 |   95.48 |      94.99     |       92.24       |   95.31
    1 |   93.41 |      93.30     | 